In [2]:
from utils import find_chunk_boundaries, TOKENIZE_PATTERN, get_pre_tokens_count

from multiprocessing import Pool
import regex as re

PARALLELIZE = True

SPECIAL_TOKENS = ["<|endoftext|>"]  # Add more special tokens as needed

filename = "./data/TinyStoriesV2-GPT4-valid.txt"
filename = "../../data/test_data.txt"

In [3]:
enable_mp = True
input_path = filename
num_process = 4

In [4]:
with open(input_path, 'rb') as f:
            boundaries = find_chunk_boundaries(f, num_process, b"<|endoftext|>")

            # only for testing purposes, limit to the first 4 chunks
            # TODO: remove this line after testing
            boundaries = boundaries[:2]

if enable_mp:
    pairs = list(zip(boundaries[:-1], boundaries[1:]))
    args = [(input_path, start, end, SPECIAL_TOKENS) for start, end in pairs]

    with Pool(processes=num_process) as pool:
            #results = pool.starmap(pre_tokenize_chunk, args)
            chunks_count = pool.starmap(get_pre_tokens_count, args)

    # combine results in a single dictionary
    pre_tokens_count = {}
    for chunk in chunks_count:
        for term, count in chunk.items():
            if term in pre_tokens_count:
                pre_tokens_count[term] += count
            else:
                pre_tokens_count[term] = count


In [5]:
pre_tokens_count

{'low': 1,
 ' low': 4,
 '\n': 2,
 'lower': 1,
 ' lower': 1,
 ' widest': 3,
 'newest': 1,
 ' newest': 5}

In [6]:
del pre_tokens_count['\n']

In [6]:
pre_tokens_count

{'low': 1,
 ' low': 4,
 'lower': 1,
 ' lower': 1,
 ' widest': 3,
 'newest': 1,
 ' newest': 5}

In [31]:
def _convert_key_to_tuple_of_bytes(key):
        """
        Convert a key to bytes.
        """
        return tuple(c.encode('utf-8') for c in key)


In [32]:
def _convert_key_to_tuple_of_bytes_v2(key):
        """
        Convert a key to bytes.
        """
        return tuple(bytes([b]) for b in key.encode('utf-8'))

In [33]:
pre_tokens_count_bytes = {_convert_key_to_tuple_of_bytes(k): v for k, v in pre_tokens_count.items()}


In [34]:
pre_tokens_count_bytes

{(b'l', b'o', b'w'): 1,
 (b' ', b'l', b'o', b'w'): 4,
 (b'l', b'o', b'w', b'e', b'r'): 1,
 (b' ', b'l', b'o', b'w', b'e', b'r'): 1,
 (b' ', b'w', b'i', b'd', b'e', b's', b't'): 3,
 (b'n', b'e', b'w', b'e', b's', b't'): 1,
 (b' ', b'n', b'e', b'w', b'e', b's', b't'): 5,
 (b'\xe2\x80\xa6',): 5}

In [10]:
def _get_pair_freq(freq_dict: dict):
        """ Calculate the frequency of each pair of consecutive bytes in the input dictionary.
        Args:
            freq_dict (dict): A dictionary where keys are tuples of bytes and values are their frequencies.

        Returns:
            dict: A dictionary with pairs of consecutive bytes as keys and their frequencies as values.
        """
        pairs_freq_dict = {}
        for key, value in freq_dict.items():
            for first, second in zip(key, key[1:]):
                pair = (first, second)
                if pair in pairs_freq_dict:
                    pairs_freq_dict[pair] += value
                else:
                    pairs_freq_dict[pair] = value

               
        return pairs_freq_dict

In [11]:
from collections import defaultdict

In [12]:
def _get_pair_dict_and_freq(freq_dict: dict):
        """ Calculate the frequency of each pair of consecutive bytes in the input dictionary.
        Args:
            freq_dict (dict): A dictionary where keys are tuples of bytes and values are their frequencies.

        Returns:
            dict: A dictionary with pairs of consecutive bytes as keys and their frequencies as values.
        """
        pair_pre_tokens_dict = defaultdict(set)
        pairs_freq_dict = {}
        for key, value in freq_dict.items():
            for first, second in zip(key, key[1:]):
                pair = (first, second)
                if pair in pairs_freq_dict:
                    pairs_freq_dict[pair] += value
                else:
                    pairs_freq_dict[pair] = value

                pair_pre_tokens_dict[pair].add(key)
        return pairs_freq_dict, pair_pre_tokens_dict

In [13]:
pre_tokens_count_bytes

{(b'l', b'o', b'w'): 1,
 (b' ', b'l', b'o', b'w'): 4,
 (b'l', b'o', b'w', b'e', b'r'): 1,
 (b' ', b'l', b'o', b'w', b'e', b'r'): 1,
 (b' ', b'w', b'i', b'd', b'e', b's', b't'): 3,
 (b'n', b'e', b'w', b'e', b's', b't'): 1,
 (b' ', b'n', b'e', b'w', b'e', b's', b't'): 5}

In [14]:
pair_freq_dict = _get_pair_freq(pre_tokens_count_bytes)
pair_freq_dict, pair_pre_tokens_dict = _get_pair_dict_and_freq(pre_tokens_count_bytes)

In [15]:
pair_pre_tokens_dict

defaultdict(set,
            {(b'l', b'o'): {(b' ', b'l', b'o', b'w'),
              (b' ', b'l', b'o', b'w', b'e', b'r'),
              (b'l', b'o', b'w'),
              (b'l', b'o', b'w', b'e', b'r')},
             (b'o', b'w'): {(b' ', b'l', b'o', b'w'),
              (b' ', b'l', b'o', b'w', b'e', b'r'),
              (b'l', b'o', b'w'),
              (b'l', b'o', b'w', b'e', b'r')},
             (b' ', b'l'): {(b' ', b'l', b'o', b'w'),
              (b' ', b'l', b'o', b'w', b'e', b'r')},
             (b'w', b'e'): {(b' ', b'l', b'o', b'w', b'e', b'r'),
              (b' ', b'n', b'e', b'w', b'e', b's', b't'),
              (b'l', b'o', b'w', b'e', b'r'),
              (b'n', b'e', b'w', b'e', b's', b't')},
             (b'e', b'r'): {(b' ', b'l', b'o', b'w', b'e', b'r'),
              (b'l', b'o', b'w', b'e', b'r')},
             (b' ', b'w'): {(b' ', b'w', b'i', b'd', b'e', b's', b't')},
             (b'w', b'i'): {(b' ', b'w', b'i', b'd', b'e', b's', b't')},
             (b'i', 

In [16]:
pair_freq_dict

{(b'l', b'o'): 7,
 (b'o', b'w'): 7,
 (b' ', b'l'): 5,
 (b'w', b'e'): 8,
 (b'e', b'r'): 2,
 (b' ', b'w'): 3,
 (b'w', b'i'): 3,
 (b'i', b'd'): 3,
 (b'd', b'e'): 3,
 (b'e', b's'): 9,
 (b's', b't'): 9,
 (b'n', b'e'): 6,
 (b'e', b'w'): 6,
 (b' ', b'n'): 5}

In [17]:
def _get_top_pair(stats):
    return max(stats, key=lambda p: (stats[p], p))

In [18]:
top_pair = _get_top_pair(pair_freq_dict)

In [19]:
top_pair

(b's', b't')

ora che ho trovato la top pair, devo fare il merge sostituendo nei pre tokens la coppia 's', 't' con 'st'

In [18]:
for pre_tokens, count in pre_tokens_count_bytes.items():
    # Your code to process each pair goes here
    break

In [19]:
pre_tokens

(b'l', b'o', b'w')

cerco solo i pre tokens che sono interessati da questa sostituzione

In [20]:
top_pair

(b's', b't')

In [20]:
pre_tokens_to_change = pair_pre_tokens_dict.get(top_pair)
pre_tokens_to_change

{(b' ', b'n', b'e', b'w', b'e', b's', b't'),
 (b' ', b'w', b'i', b'd', b'e', b's', b't'),
 (b'n', b'e', b'w', b'e', b's', b't')}

# funzione per modificare il pre token inserendo la top pair

In [21]:
def _merge_pair_in_token(pair, token):
    first, second = pair
    merged_token = []
    i = 0
    while i < len(token):
        if i < len(token) - 1 and token[i] == first and token[i + 1] == second:
            merged_token.append(first + second)
            i += 2
        else:
            merged_token.append(token[i])
            i += 1
    return tuple(merged_token)

In [22]:
test_tuple = (b'a', b's', b't', b'a', b'r', b's', b't') 
test_top_pair = (b's', b't')
assert _merge_pair_in_token(test_top_pair, test_tuple) == (b'a', b'st', b'a', b'r', b'st')
# Expected output: (b'a', b'st', b'a', b'r', b'st')

In [23]:
test_top_pair = (b'a', b's')
assert _merge_pair_in_token(test_top_pair, test_tuple) == (b'as', b't', b'a', b'r', b's', b't')
# Expected output: (b'ast', b'a', b'r', b'st')


applico la sostituzione

In [25]:
pre_tokens_count_bytes

{(b'l', b'o', b'w'): 1,
 (b' ', b'l', b'o', b'w'): 4,
 (b'\n',): 2,
 (b'l', b'o', b'w', b'e', b'r'): 1,
 (b' ', b'l', b'o', b'w', b'e', b'r'): 1,
 (b' ', b'w', b'i', b'd', b'e', b's', b't'): 3,
 (b'n', b'e', b'w', b'e', b's', b't'): 1,
 (b' ', b'n', b'e', b'w', b'e', b's', b't'): 5}

In [26]:
pre_tokens_to_change

{(b' ', b'n', b'e', b'w', b'e', b's', b't'),
 (b' ', b'w', b'i', b'd', b'e', b's', b't'),
 (b'n', b'e', b'w', b'e', b's', b't')}

# creo il nuovo dict per il conteggio dei pre tokens

In [27]:
pre_tokens_to_change

{(b' ', b'n', b'e', b'w', b'e', b's', b't'),
 (b' ', b'w', b'i', b'd', b'e', b's', b't'),
 (b'n', b'e', b'w', b'e', b's', b't')}

In [28]:
top_pair

(b's', b't')

In [29]:
??_get_pair_freq

Signature: _get_pair_freq(freq_dict: dict)
Source:   
def _get_pair_freq(freq_dict: dict):
        """ Calculate the frequency of each pair of consecutive bytes in the input dictionary.
        Args:
            freq_dict (dict): A dictionary where keys are tuples of bytes and values are their frequencies.

        Returns:
            dict: A dictionary with pairs of consecutive bytes as keys and their frequencies as values.
        """
        pairs_freq_dict = {}
        for key, value in freq_dict.items():
            for first, second in zip(key, key[1:]):
                pair = (first, second)
                if pair in pairs_freq_dict:
                    pairs_freq_dict[pair] += value
                else:
                    pairs_freq_dict[pair] = value


        return pairs_freq_dict
File:      /var/folders/4t/s4gbk8dd0gj_dyfscktscncw0000gn/T/ipykernel_52708/3919515239.py
Type:      function

In [24]:
def _get_pair_from_token(token):
    pair_list = []
    for i in range(len(token) - 1):
        pair_list.append((token[i], token[i + 1]))
    return pair_list

In [25]:
update_pair_freq_dict = pair_freq_dict.copy()

In [26]:
sorted(pair_pre_tokens_dict.keys())

[(b' ', b'l'),
 (b' ', b'n'),
 (b' ', b'w'),
 (b'd', b'e'),
 (b'e', b'r'),
 (b'e', b's'),
 (b'e', b'w'),
 (b'i', b'd'),
 (b'l', b'o'),
 (b'n', b'e'),
 (b'o', b'w'),
 (b's', b't'),
 (b'w', b'e'),
 (b'w', b'i')]

In [ ]:
new_pre_tokens_count_bytes = pre_tokens_count_bytes.copy()
new_pair_pre_tokens_dict = pair_pre_tokens_dict.copy()

for pre_tokens in pre_tokens_to_change:
    new_pre_tokens = _merge_pair_in_token(top_pair, pre_tokens)
    old_count = pre_tokens_count_bytes.get(pre_tokens)
    if new_pre_tokens in new_pre_tokens_count_bytes:
        new_pre_tokens_count_bytes[new_pre_tokens] += old_count
    else:
        new_pre_tokens_count_bytes[new_pre_tokens] = old_count

    # get the pair from pre_tokens
    old_pair_list = _get_pair_from_token(pre_tokens)
    for pair in old_pair_list:
        # decrement the frequency of the old pair by the old count
        update_pair_freq_dict[pair] = update_pair_freq_dict.get(pair, 0) - old_count

        # remove old pre tokens
        pre_tokens_list = list(new_pair_pre_tokens_dict.get(pair))
        if pre_tokens_list:
            pre_tokens_list.remove(pre_tokens)

        new_pair_pre_tokens_dict[pair] = set(pre_tokens_list)

    new_pair_list = _get_pair_from_token(new_pre_tokens)
    for pair in new_pair_list:
        update_pair_freq_dict[pair] = update_pair_freq_dict.get(pair, 0) + old_count

        # add new pre tokens
        pre_tokens_list = list(new_pair_pre_tokens_dict.get(pair))
        if pre_tokens_list:
            pre_tokens_list.append(new_pre_tokens)
        else:
            pre_tokens_list = [new_pre_tokens]

        new_pair_pre_tokens_dict[pair] = set(pre_tokens_list)
    del new_pre_tokens_count_bytes[pre_tokens]


# remove pair with zero frequency
pairs_to_remove = [pair for pair, freq in update_pair_freq_dict.items() if freq == 0]
for pair in pairs_to_remove:
    del update_pair_freq_dict[pair]



TypeError: 'NoneType' object is not iterable

In [37]:
pair

(b'e', b'st')

In [35]:
new_pair_pre_tokens_dict

defaultdict(set,
            {(b'l', b'o'): {(b' ', b'l', b'o', b'w'),
              (b' ', b'l', b'o', b'w', b'e', b'r'),
              (b'l', b'o', b'w'),
              (b'l', b'o', b'w', b'e', b'r')},
             (b'o', b'w'): {(b' ', b'l', b'o', b'w'),
              (b' ', b'l', b'o', b'w', b'e', b'r'),
              (b'l', b'o', b'w'),
              (b'l', b'o', b'w', b'e', b'r')},
             (b' ', b'l'): {(b' ', b'l', b'o', b'w'),
              (b' ', b'l', b'o', b'w', b'e', b'r')},
             (b'w', b'e'): {(b' ', b'l', b'o', b'w', b'e', b'r'),
              (b' ', b'n', b'e', b'w', b'e', b'st'),
              (b'l', b'o', b'w', b'e', b'r'),
              (b'n', b'e', b'w', b'e', b's', b't')},
             (b'e', b'r'): {(b' ', b'l', b'o', b'w', b'e', b'r'),
              (b'l', b'o', b'w', b'e', b'r')},
             (b' ', b'w'): {(b' ', b'w', b'i', b'd', b'e', b's', b't')},
             (b'w', b'i'): {(b' ', b'w', b'i', b'd', b'e', b's', b't')},
             (b'i', b'd')

In [101]:
pair_pre_tokens_dict.get(top_pair)

{(b' ', b'n', b'e', b'w', b'e', b's', b't'),
 (b' ', b'w', b'i', b'd', b'e', b's', b't'),
 (b'n', b'e', b'w', b'e', b's', b't')}

In [59]:
temp = pair_pre_tokens_dict[(b'i', b'd')]

In [61]:
temp.remove((b' ', b'w', b'i', b'd', b'e', b's', b't'))

In [62]:
temp

set()

In [50]:
list(pre_tokens_to_change)[0]

(b' ', b'w', b'i', b'd', b'e', b's', b't')

In [51]:
old_pair_list = _get_pair_from_token(list(pre_tokens_to_change)[0])

In [52]:
old_pair_list

[(b' ', b'w'),
 (b'w', b'i'),
 (b'i', b'd'),
 (b'd', b'e'),
 (b'e', b's'),
 (b's', b't')]

In [56]:
pair_pre_tokens_dict[(b'i', b'd')]

{(b' ', b'w', b'i', b'd', b'e', b's', b't')}

In [ ]:
pair_pre_tokens_dict

defaultdict(set,
            {(b'l', b'o'): {(b' ', b'l', b'o', b'w'),
              (b' ', b'l', b'o', b'w', b'e', b'r'),
              (b'l', b'o', b'w'),
              (b'l', b'o', b'w', b'e', b'r')},
             (b'o', b'w'): {(b' ', b'l', b'o', b'w'),
              (b' ', b'l', b'o', b'w', b'e', b'r'),
              (b'l', b'o', b'w'),
              (b'l', b'o', b'w', b'e', b'r')},
             (b' ', b'l'): {(b' ', b'l', b'o', b'w'),
              (b' ', b'l', b'o', b'w', b'e', b'r')},
             (b'w', b'e'): {(b' ', b'l', b'o', b'w', b'e', b'r'),
              (b' ', b'n', b'e', b'w', b'e', b's', b't'),
              (b'l', b'o', b'w', b'e', b'r'),
              (b'n', b'e', b'w', b'e', b's', b't')},
             (b'e', b'r'): {(b' ', b'l', b'o', b'w', b'e', b'r'),
              (b'l', b'o', b'w', b'e', b'r')},
             (b' ', b'w'): {(b' ', b'w', b'i', b'd', b'e', b's', b't')},
             (b'w', b'i'): {(b' ', b'w', b'i', b'd', b'e', b's', b't')},
             (b'i', 

In [36]:
# make a test, compute update pair frequency dictionary iterating over new_pre_tokens_count_bytes
update_pair_freq_dict_test = _get_pair_freq(new_pre_tokens_count_bytes)

In [37]:
assert update_pair_freq_dict == update_pair_freq_dict_test

# Create some functions

In [27]:
def update_frequency_dict(pair_freq_dict, pre_tokens_count_bytes, pair_pre_tokens_dict, top_pair, pre_tokens_to_change):
    new_pre_tokens_count_bytes = pre_tokens_count_bytes.copy()
    new_pair_freq_dict = pair_freq_dict.copy()
    new_pair_pre_tokens_dict = pair_pre_tokens_dict.copy()

    for pre_tokens in pre_tokens_to_change:
        new_pre_tokens = _merge_pair_in_token(top_pair, pre_tokens)
        old_count = pre_tokens_count_bytes.get(pre_tokens)
        if new_pre_tokens in new_pre_tokens_count_bytes:
            new_pre_tokens_count_bytes[new_pre_tokens] += old_count
        else:
            new_pre_tokens_count_bytes[new_pre_tokens] = old_count

        # get the pair from pre_tokens
        old_pair_list = _get_pair_from_token(pre_tokens)
        for pair in old_pair_list:
            new_pair_freq_dict[pair] = new_pair_freq_dict.get(pair, 0) - old_count
            # remove old pre tokens from the pair_pre_tokens_dict
            pre_tokens_list = list(new_pair_pre_tokens_dict.get(pair))
            if pre_tokens_list:
                pre_tokens_list.remove(pre_tokens)
            new_pair_pre_tokens_dict[pair] = set(pre_tokens_list)

        new_pair_list = _get_pair_from_token(new_pre_tokens)
        for pair in new_pair_list:
            new_pair_freq_dict[pair] = new_pair_freq_dict.get(pair, 0) + old_count
            # add new pre tokens to the pair_pre_tokens_dict
            if pair not in new_pair_pre_tokens_dict:
                new_pair_pre_tokens_dict[pair] = set()
            pre_tokens_list = list(new_pair_pre_tokens_dict.get(pair))
            if pre_tokens_list:
                pre_tokens_list.append(new_pre_tokens)
            else:
                pre_tokens_list = [new_pre_tokens]
            new_pair_pre_tokens_dict[pair] = set(pre_tokens_list)
        del new_pre_tokens_count_bytes[pre_tokens]

    return new_pair_freq_dict, new_pre_tokens_count_bytes, new_pair_pre_tokens_dict

In [32]:
def update_frequency_dict_v2(pair_freq_dict, pre_tokens_count_bytes, pair_pre_tokens_dict, top_pair, pre_tokens_to_change):
    new_pre_tokens_count_bytes = pre_tokens_count_bytes.copy()
    new_pair_freq_dict = pair_freq_dict.copy()
    new_pair_pre_tokens_dict = pair_pre_tokens_dict.copy()

    for pre_tokens in pre_tokens_to_change:
        new_pre_tokens = _merge_pair_in_token(top_pair, pre_tokens)
        old_count = pre_tokens_count_bytes.get(pre_tokens)
        if new_pre_tokens in new_pre_tokens_count_bytes:
            new_pre_tokens_count_bytes[new_pre_tokens] += old_count
        else:
            new_pre_tokens_count_bytes[new_pre_tokens] = old_count

        # get the pair from pre_tokens
        old_pair_list = _get_pair_from_token(pre_tokens)
        for pair in old_pair_list:
            new_pair_freq_dict[pair] = new_pair_freq_dict.get(pair, 0) - old_count

            if new_pair_freq_dict[pair] <= 0:
                del new_pair_freq_dict[pair]
                del new_pair_pre_tokens_dict[pair]
            else:
                # remove old pre tokens from the pair_pre_tokens_dict
                pre_tokens_list = list(new_pair_pre_tokens_dict.get(pair))
                if pre_tokens_list:
                    pre_tokens_list.remove(pre_tokens)
                new_pair_pre_tokens_dict[pair] = set(pre_tokens_list)

        new_pair_list = _get_pair_from_token(new_pre_tokens)
        for pair in new_pair_list:
            new_pair_freq_dict[pair] = new_pair_freq_dict.get(pair, 0) + old_count
            # add new pre tokens to the pair_pre_tokens_dict
            if pair not in new_pair_pre_tokens_dict:
                new_pair_pre_tokens_dict[pair] = set()
            pre_tokens_list = list(new_pair_pre_tokens_dict.get(pair))
            if pre_tokens_list:
                pre_tokens_list.append(new_pre_tokens)
            else:
                pre_tokens_list = [new_pre_tokens]
            new_pair_pre_tokens_dict[pair] = set(pre_tokens_list)
        del new_pre_tokens_count_bytes[pre_tokens]

    return new_pair_freq_dict, new_pre_tokens_count_bytes, new_pair_pre_tokens_dict

In [28]:
new_pair_freq_dict, new_pre_tokens_count_bytes, new_pair_pre_tokens_dict = update_frequency_dict(pair_freq_dict, pre_tokens_count_bytes, pair_pre_tokens_dict, top_pair, pre_tokens_to_change)

In [33]:
new_pair_freq_dict, new_pre_tokens_count_bytes, new_pair_pre_tokens_dict = update_frequency_dict_v2(pair_freq_dict, pre_tokens_count_bytes, pair_pre_tokens_dict, top_pair, pre_tokens_to_change)

In [34]:
new_pair_freq_dict, new_pre_tokens_count_bytes, new_pair_pre_tokens_dict

({(b'l', b'o'): 7,
  (b'o', b'w'): 7,
  (b' ', b'l'): 5,
  (b'w', b'e'): 8,
  (b'e', b'r'): 2,
  (b'n', b'e'): 6,
  (b'e', b'w'): 6,
  (b' ', b'w'): 3,
  (b'w', b'i'): 3,
  (b'i', b'd'): 3,
  (b'd', b'e'): 3,
  (b'e', b'st'): 9,
  (b' ', b'n'): 5},
 {(b'l', b'o', b'w'): 1,
  (b' ', b'l', b'o', b'w'): 4,
  (b'l', b'o', b'w', b'e', b'r'): 1,
  (b' ', b'l', b'o', b'w', b'e', b'r'): 1,
  (b' ', b'w', b'i', b'd', b'e', b'st'): 3,
  (b' ', b'n', b'e', b'w', b'e', b'st'): 5,
  (b'n', b'e', b'w', b'e', b'st'): 1},
 defaultdict(set,
             {(b'l', b'o'): {(b' ', b'l', b'o', b'w'),
               (b' ', b'l', b'o', b'w', b'e', b'r'),
               (b'l', b'o', b'w'),
               (b'l', b'o', b'w', b'e', b'r')},
              (b'o', b'w'): {(b' ', b'l', b'o', b'w'),
               (b' ', b'l', b'o', b'w', b'e', b'r'),
               (b'l', b'o', b'w'),
               (b'l', b'o', b'w', b'e', b'r')},
              (b' ', b'l'): {(b' ', b'l', b'o', b'w'),
               (b' ', b'l', b'o',

In [9]:
_convert_key_to_tuple_of_bytes

<function __main__._convert_key_to_tuple_of_bytes(key)>

In [10]:
pre_tokens_count

{'low': 1,
 ' low': 4,
 'lower': 1,
 ' lower': 1,
 ' widest': 3,
 'newest': 1,
 ' newest': 5}

In [ ]:
pre_tokens_count

In [11]:
b'\xe2\x80\xa6'.decode('utf-8')

'…'

In [ ]:
'…'.encode('utf-8')

b'\xe2\x80\xa6'

In [15]:
pre_tokens_count['…'] = 5

In [36]:
pre_tokens_count_bytes = {_convert_key_to_tuple_of_bytes(k): v for k, v in pre_tokens_count.items()}
pre_tokens_count_bytes

{(b'l', b'o', b'w'): 1,
 (b' ', b'l', b'o', b'w'): 4,
 (b'l', b'o', b'w', b'e', b'r'): 1,
 (b' ', b'l', b'o', b'w', b'e', b'r'): 1,
 (b' ', b'w', b'i', b'd', b'e', b's', b't'): 3,
 (b'n', b'e', b'w', b'e', b's', b't'): 1,
 (b' ', b'n', b'e', b'w', b'e', b's', b't'): 5,
 (b'\xe2\x80\xa6',): 5}

In [35]:
pre_tokens_count_bytes = {_convert_key_to_tuple_of_bytes_v2(k): v for k, v in pre_tokens_count.items()}
pre_tokens_count_bytes

{(b'l', b'o', b'w'): 1,
 (b' ', b'l', b'o', b'w'): 4,
 (b'l', b'o', b'w', b'e', b'r'): 1,
 (b' ', b'l', b'o', b'w', b'e', b'r'): 1,
 (b' ', b'w', b'i', b'd', b'e', b's', b't'): 3,
 (b'n', b'e', b'w', b'e', b's', b't'): 1,
 (b' ', b'n', b'e', b'w', b'e', b's', b't'): 5,
 (b'\xe2', b'\x80', b'\xa6'): 5}

In [19]:
tuple(c.encode('utf-8') for c in '…')

(b'\xe2\x80\xa6',)

In [20]:
list('…')

['…']

In [21]:
list('abc')

['a', 'b', 'c']

In [22]:
key='…'

In [23]:
tuple(bytes([b]) for b in key.encode('utf-8'))   # encode whole string, then per-byte

(b'\xe2', b'\x80', b'\xa6')